In [0]:
CATALOG = "workspace"           
SCHEMA = "late_txn_pipeline"    


VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/pipeline_data"

LANDING_PATH = f"{VOLUME_ROOT}/landing/sat_act"         
SCHEMA_LOCATION = f"{VOLUME_ROOT}/_autoloader_schema"    
CHECKPOINT_BRONZE = f"{VOLUME_ROOT}/_checkpoints/bronze" 

BRONZE_PATH = f"{VOLUME_ROOT}/bronze/sales"
SILVER_PATH = f"{VOLUME_ROOT}/silver/sales"
GOLD_PATH = f"{VOLUME_ROOT}/gold/revenue"
QUARANTINE_PATH = f"{VOLUME_ROOT}/quarantine/sales"
WATERMARK_TABLE = f"{CATALOG}.{SCHEMA}.watermark"


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.pipeline_data")

dbutils.fs.mkdirs(LANDING_PATH)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {WATERMARK_TABLE} (
        table_name STRING,
        last_processed_date DATE
    )
""")

print(f"Landing path ready: {LANDING_PATH}")
print("Volume contents:")
display(dbutils.fs.ls(VOLUME_ROOT))
print("Landing folder contents (upload your CSV(s) here):")
display(dbutils.fs.ls(LANDING_PATH))

Landing path ready: /Volumes/workspace/late_txn_pipeline/pipeline_data/landing/sat_act
Volume contents:


path,name,size,modificationTime
dbfs:/Volumes/workspace/late_txn_pipeline/pipeline_data/_autoloader_schema/,_autoloader_schema/,0,1785639197787
dbfs:/Volumes/workspace/late_txn_pipeline/pipeline_data/_checkpoints/,_checkpoints/,0,1785639197787
dbfs:/Volumes/workspace/late_txn_pipeline/pipeline_data/bronze/,bronze/,0,1785639197787
dbfs:/Volumes/workspace/late_txn_pipeline/pipeline_data/gold/,gold/,0,1785639197787
dbfs:/Volumes/workspace/late_txn_pipeline/pipeline_data/landing/,landing/,0,1785639197787
dbfs:/Volumes/workspace/late_txn_pipeline/pipeline_data/silver/,silver/,0,1785639197787


Landing folder contents (upload your CSV(s) here):


path,name,size,modificationTime
dbfs:/Volumes/workspace/late_txn_pipeline/pipeline_data/landing/sat_act/sales_2000_rows.csv,sales_2000_rows.csv,70585,1785520310000


In [0]:
from pyspark.sql.functions import col, sum as spark_sum, to_date
from pyspark.sql import Window
from pyspark.sql.functions import row_number


landing_files = [f for f in dbutils.fs.ls(LANDING_PATH) if f.name.endswith(".csv")]

if not landing_files:
    print(f"No CSV files found in {LANDING_PATH} yet. "
          f"Upload your source file(s) there and re-run this cell.")
else:
    bronze_stream = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("pathGlobFilter", "*.csv")
        .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
        .option("cloudFiles.inferColumnTypes", "true")
        .load(LANDING_PATH)
    )

    (
        bronze_stream.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT_BRONZE)
        .outputMode("append")
        .trigger(availableNow=True)   
        .start(BRONZE_PATH)
        .awaitTermination()
    )

    bronze_df = spark.read.format("delta").load(BRONZE_PATH)
    print(f"Bronze rows: {bronze_df.count()}")
    display(bronze_df.limit(5))

Bronze rows: 2000


txn_id,user_id,txn_date,amount,ingestion_date,_rescued_data
1,427,2024-01-08,2353,2024-01-08,null
2,225,2024-01-15,939,2024-01-15,null
3,446,2024-02-17,812,2024-02-27,null
4,402,2024-01-28,344,2024-01-28,null
5,147,2024-01-14,4239,2024-01-15,null


In [0]:
raw = spark.read.format("delta").load(BRONZE_PATH)


dedup_window = Window.partitionBy("txn_id").orderBy(col("ingestion_date").asc())

silver_df = (
    raw.withColumn("txn_date", to_date(col("txn_date")))
    .withColumn("ingestion_date", to_date(col("ingestion_date")))
    .withColumn("_rn", row_number().over(dedup_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
    .filter(col("amount") > 0)
)

(
    silver_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

print(f"Silver rows after cleansing: {silver_df.count()}")

Silver rows after cleansing: 2000


In [0]:
null_txn_id_df = raw.filter(col("txn_id").isNull())
null_txn_id_count = null_txn_id_df.count()
if null_txn_id_count > 0:
    (
        null_txn_id_df.write.format("delta")
        .mode("append")
        .save(QUARANTINE_PATH)
    )
print(f"Null txn_id records quarantined: {null_txn_id_count}")

dup_txn_ids = raw.groupBy("txn_id").count().filter("count > 1")
print(f"Duplicate txn_id groups found (pre-dedup): {dup_txn_ids.count()}")

negative_amounts = raw.filter(col("amount") < 0)
print(f"Negative amount records flagged for review: {negative_amounts.count()}")

Null txn_id records quarantined: 0
Duplicate txn_id groups found (pre-dedup): 0
Negative amount records flagged for review: 0


In [0]:
silver = spark.read.format("delta").load(SILVER_PATH)

gold_df = silver.groupBy("txn_date").agg(spark_sum("amount").alias("daily_revenue"))

(
    gold_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("txn_date")
    .save(GOLD_PATH)
)

print(f"Gold rows (distinct dates): {gold_df.count()}")

Gold rows (distinct dates): 60


In [0]:
late_df = silver.filter(col("ingestion_date") > col("txn_date"))
late_count = late_df.count()
total_count = silver.count()

late_pct = (late_count / total_count) if total_count > 0 else 0.0
print(f"Late transactions: {late_count} / {total_count} ({late_pct:.1%})")
display(late_df.select("txn_id", "txn_date", "ingestion_date", "amount").limit(10))

Late transactions: 1415 / 2000 (70.8%)


txn_id,txn_date,ingestion_date,amount
3,2024-02-17,2024-02-27,812
5,2024-01-14,2024-01-15,4239
6,2024-01-02,2024-01-12,1728
7,2024-02-11,2024-02-21,3536
8,2024-01-29,2024-02-13,2378
12,2024-01-07,2024-01-10,2917
14,2024-01-08,2024-01-13,745
15,2024-01-19,2024-02-03,3062
17,2024-01-15,2024-01-17,753
19,2024-01-30,2024-02-02,1432


In [0]:
from delta.tables import DeltaTable


affected_dates = late_df.select("txn_date").distinct()
affected_dates_count = affected_dates.count()
print(f"Distinct affected dates: {affected_dates_count}")

if affected_dates_count == 0:
    print("No late transactions detected — Gold layer already reflects all data. Skipping MERGE.")
else:
   
    recompute_df = (
        silver.join(affected_dates, "txn_date")
        .groupBy("txn_date")
        .agg(spark_sum("amount").alias("daily_revenue"))
    )

 
    gold_table = DeltaTable.forPath(spark, GOLD_PATH)

    (
        gold_table.alias("t")
        .merge(recompute_df.alias("s"), "t.txn_date = s.txn_date")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("MERGE complete: affected historical dates corrected in Gold.")

display(spark.read.format("delta").load(GOLD_PATH).orderBy("txn_date"))

Distinct affected dates: 60
MERGE complete: affected historical dates corrected in Gold.


txn_date,daily_revenue
2024-01-01,56799
2024-01-02,66741
2024-01-03,93732
2024-01-04,70961
2024-01-05,69963
2024-01-06,94410
2024-01-07,95732
2024-01-08,108769
2024-01-09,88910
2024-01-10,69564


In [0]:

display(spark.sql(f"DESCRIBE HISTORY delta.`{GOLD_PATH}`"))



version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-08-02T02:54:10.000Z,77086457722183,ingalevarad34@gmail.com,MERGE,"Map(predicate -> [""(txn_date#11699 = txn_date#11537)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1722798920827607),d16573f0-c85b-4872-8e8b-4f31e9581dcd,0802-025257-24hfuiuk-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 60, numTargetBytesAdded -> 64435, numTargetBytesRemoved -> 64435, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 60, executionTimeMs -> 8845, materializeSourceTimeMs -> 1176, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3757, numTargetRowsUpdated -> 60, numOutputRows -> 60, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 60, numTargetFilesRemoved -> 60, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3757)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-08-02T02:53:57.000Z,77086457722183,ingalevarad34@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [""txn_date""], canOverwriteSchema -> true)",null,List(1722798920827607),673222b2-74b9-4a53-9825-889bbf949b72,0802-025257-24hfuiuk-v2n,3,WriteSerializable,false,"Map(numFiles -> 60, numRemovedFiles -> 60, numRemovedBytes -> 64435, numDeletionVectorsRemoved -> 0, numOutputRows -> 60, numOutputBytes -> 64435)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-08-01T15:56:21.000Z,77086457722183,ingalevarad34@gmail.com,MERGE,"Map(predicate -> [""(txn_date#11699 = txn_date#11537)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1722798920827607),f4fdab51-ccee-47a0-b5de-51b23090ed68,0801-155508-pimvvzkd-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 60, numTargetBytesAdded -> 64435, numTargetBytesRemoved -> 64435, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 60, executionTimeMs -> 8374, materializeSourceTimeMs -> 1103, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3355, numTargetRowsUpdated -> 60, numOutputRows -> 60, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 60, numTargetFilesRemoved -> 60, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3767)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-08-01T15:56:08.000Z,77086457722183,ingalevarad34@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [""txn_date""], canOverwriteSchema -> true)",null,List(1722798920827607),2a067ec4-9597-4573-9625-3a16b1a0a20e,0801-155508-pimvvzkd-v2n,1,WriteSerializable,false,"Map(numFiles -> 60, numRemovedFiles -> 60, numRemovedBytes -> 64435, numDeletionVectorsRemoved -> 0, numOutputRows -> 60, numOutputBytes -> 64435)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-08-01T03:15:58.000Z,77086457722183,ingalevarad34@gmail.com,MERGE,"Map(predicate -> [""(txn_date#11897 = txn_date#11711)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1722798920827607),632f60a7-d724-4314-9d57-9958b43987fc,0801-030824-5o8jgc0q-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0,

In [0]:
last_processed_date = silver.agg({"txn_date": "max"}).collect()[0][0]

if last_processed_date is None:
    print("Silver layer is empty — skipping watermark update.")
else:
    spark.sql(f"""
        MERGE INTO {WATERMARK_TABLE} t
        USING (SELECT 'sales' AS table_name, DATE('{last_processed_date}') AS last_processed_date) s
        ON t.table_name = s.table_name
        WHEN MATCHED THEN UPDATE SET t.last_processed_date = s.last_processed_date
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Watermark updated. Last processed date: {last_processed_date}")

Watermark updated. Last processed date: 2024-02-29


In [0]:
print("Pipeline run complete.")
print(f"  Bronze rows:        {raw.count()}")
print(f"  Silver rows:        {silver.count()}")
print(f"  Gold dates (total): {spark.read.format('delta').load(GOLD_PATH).count()}")
print(f"  Late transactions:  {late_count} ({late_pct:.1%})")
print(f"  Affected dates:     {affected_dates_count}")

Pipeline run complete.
  Bronze rows:        2000
  Silver rows:        2000
  Gold dates (total): 60
  Late transactions:  1415 (70.8%)
  Affected dates:     60
